
> **FLOPs 是“计算量”，FLOPS 是“计算速度”。**  
> 小写 s 和大写 S 经常被混用，但含义不同。

---

# 1. 最基础：什么是浮点运算？

计算机里的数分很多种：

- 整数：`1, 2, -3`
- 浮点数：`1.23, -0.005, 3.14159`

浮点数就是带小数点的数，用科学计数法表示，比如：

$$
3.14159 = 3.14159 \times 10^0
$$

神经网络里的权重、激活值通常是浮点数，比如 FP32、FP16、BF16。

浮点运算就是：

- 浮点加法：$a + b$
- 浮点乘法：$a \times b$
- 浮点除法：$a / b$
- 浮点开方、指数、对数等

**每一次这样的运算，都算一次 FLOP。**

---

# 2. FLOPs 和 FLOPS 的区别

这是最容易混的地方。

| 写法 | 全称 | 含义 | 单位 | 例子 |
|---|---|---|---|---|
| **FLOPs** | Floating Point Operations | 浮点运算**次数** | 次 | 4.1 GFLOPs |
| **FLOPS** | Floating Point Operations Per Second | 每秒浮点运算**次数** | 次/秒 | 100 TFLOPS |

所以：

- 模型论文里说 “ResNet-50 有 4.1 GFLOPs”，指的是**计算量**。
- 显卡参数里说 “A100 有 312 TFLOPS”，指的是**算力速度**。

很多地方大小写不严格，但你要能根据上下文判断。

---

# 3. 为什么需要 FLOPs？

因为我们要衡量：

1. **模型有多复杂**：需要多少次计算。
2. **硬件够不够快**：GPU 每秒能做多少次计算。
3. **推理要多久**：理论时间 ≈ 总 FLOPs / 硬件 FLOPS。
4. **训练成本多大**：训练还要反向传播，计算量更大。
5. **模型能不能部署到手机**：手机算力有限。

比如：

- 手机 NPU 可能只有几 TFLOPS。
- 如果模型需要 100 GFLOPs，理论最快也要：

$$
100 \text{ GFLOPs} / 1 \text{ TFLOPS} = 0.1 \text{ 秒}
$$

当然实际会更慢，因为还有内存、带宽、调度等开销。

---

# 4. 基本运算怎么计数？

## 4.1 加法、乘法、乘加

一次乘法：

$$
y = a \times b
$$

算 **1 FLOP**。

一次加法：

$$
y = a + b
$$

也算 **1 FLOP**。

所以：

$$
y = a \times b + c
$$

算 **2 FLOPs**：一次乘法 + 一次加法。

## 4.2 MACs：乘加运算

神经网络里最常见的是：

$$
y = w \cdot x + b
$$

这其实是一次乘法加一次加法，叫做 **MAC**：

> MAC = Multiply-Accumulate，乘加运算。

通常规定：

$$
1 \text{ MAC} = 2 \text{ FLOPs}
$$

所以：

$$
\text{FLOPs} = 2 \times \text{MACs}
$$

有些工具只统计 MACs，有些统计 FLOPs，结果会差 2 倍。

## 4.3 点积

比如：

$$
y = w_1x_1 + w_2x_2 + w_3x_3
$$

计算过程：

- 3 次乘法：$w_1x_1, w_2x_2, w_3x_3$
- 2 次加法：先加前两个，再加第三个

总共：

$$
3 + 2 = 5 \text{ FLOPs}
$$

如果按 MAC 算：

- 3 个乘加，每个 2 FLOPs，共 6 FLOPs。

两者略有差异，通常忽略加法次数少 1 的问题。

---

# 5. 线性层 / 全连接层怎么算？

设：

- 输入维度：$I$
- 输出维度：$O$
- batch size：$B$

## 5.1 无 bias

$$
y = Wx
$$

权重矩阵 $W$ 大小：

$$
O \times I
$$

参数量：

$$
|W| = I \times O
$$

每个输出元素需要：

- $I$ 次乘法
- $I-1$ 次加法

所以每个输出约：

$$
2I - 1 \text{ FLOPs}
$$

总输出有 $O$ 个，batch 为 $B$：

$$
\text{FLOPs} = B \times O \times (2I - 1)
\approx 2BIO
$$

而权重参数量是 $IO$，所以：

$$
\text{FLOPs} \approx 2 \times B \times |W|
$$

如果按 MAC 算：

$$
\text{MACs} = B \times I \times O = B \times |W|
$$

## 5.2 有 bias

$$
y = Wx + b
$$

每个输出最后再加一次 bias：

$$
2I \text{ FLOPs}
$$

总：

$$
\text{FLOPs} = 2BIO
$$

所以：

> 单样本、有 bias 或无 bias 忽略最后加法时，线性层 FLOPs ≈ 2 × 权重矩阵参数量。

如果 batch 是 $B$，再乘 $B$。

---

# 6. 卷积层怎么算？

卷积是神经网络里计算量最大的部分。

设：

- 输入通道：$C_{in}$
- 输出通道：$C_{out}$
- 卷积核：$K_h \times K_w$
- 输出尺寸：$H_{out} \times W_{out}$
- batch size：$B$

## 6.1 普通卷积

每个输出元素需要：

$$
C_{in} \times K_h \times K_w
$$

次乘加。

输出元素总数：

$$
B \times H_{out} \times W_{out} \times C_{out}
$$

所以：

$$
\text{MACs} = B \times H_{out} \times W_{out} \times C_{out} \times C_{in} \times K_h \times K_w
$$

$$
\text{FLOPs} = 2 \times \text{MACs}
$$

即：

$$
\text{FLOPs} = 2 B H_{out} W_{out} C_{out} C_{in} K_h K_w
$$

## 6.2 分组卷积

如果分成 $g$ 组：

$$
\text{MACs} = B H_{out} W_{out} C_{out} \times \frac{C_{in}}{g} \times K_h K_w
$$

FLOPs 再乘 2。

## 6.3 深度可分离卷积

分两步：

**Depthwise 卷积：**

$$
\text{MACs} = B H_{out} W_{out} C_{in} K_h K_w
$$

**Pointwise 卷积（1×1）：**

$$
\text{MACs} = B H_{out} W_{out} C_{out} C_{in}
$$

总 MACs 相加，FLOPs 再乘 2。

深度可分离卷积能大幅降低计算量，所以 MobileNet 系列很快。

---

# 7. 其他层的计算

## 7.1 激活函数

ReLU：

$$
y = \max(0, x)
$$

每个元素算 1 次比较或 1 FLOP，通常很小，很多统计工具忽略。

Sigmoid、Tanh 包含指数运算，计算量比 ReLU 大，但在大模型中占比通常不高。

## 7.2 池化

最大池化：每个输出元素做几次比较，通常忽略或按 1 FLOP 估。

平均池化：每个输出元素做加法 + 除法，计算量也小。

## 7.3 归一化

BatchNorm、LayerNorm：

- 每个元素减均值、除标准差、乘 gamma、加 beta。
- 大约每个元素 4~5 FLOPs。
- 相比卷积很小，但 Transformer 中 LayerNorm 也不能完全忽略。

## 7.4 Softmax

每个元素要做指数、求和、除法，计算量比普通加法大，但通常占比小。

## 7.5 注意力机制

Transformer 的注意力计算量很大。

设：

- batch：$B$
- 序列长度：$N$
- 模型维度：$d$
- 头数：$H$，每头维度 $d_h = d/H$

主要部分：

1. QKV 线性投影：

$$
3 \times 2 B N d^2 = 6 B N d^2
$$

2. $QK^T$：

$$
2 B N^2 d
$$

3. Softmax：约 $B H N^2$，通常忽略。

4. 加权和 $AV$：

$$
2 B N^2 d
$$

5. 输出投影：

$$
2 B N d^2
$$

所以注意力部分大约：

$$
8 B N d^2 + 4 B N^2 d
$$

FFN 部分通常是两个线性层：

$$
2 B N d d_{ff} + 2 B N d_{ff} d = 4 B N d d_{ff}
$$

如果 $d_{ff} = 4d$，则：

$$
16 B N d^2
$$

所以 Transformer 每层计算量约：

$$
24 B N d^2 + 4 B N^2 d
$$

当 $N$ 很大时，$N^2$ 项会主导。

---

# 8. 单位换算

FLOPs 常用单位：

| 单位 | 数值 |
|---|---|
| 1 KFLOPs | $10^3$ |
| 1 MFLOPs | $10^6$ |
| 1 GFLOPs | $10^9$ |
| 1 TFLOPs | $10^{12}$ |
| 1 PFLOPs | $10^{15}$ |

注意：

- **FLOPs** 是次数，单位就是“次”。
- **FLOPS** 是速度，单位是“次/秒”。

比如：

- 模型计算量：4.1 GFLOPs = $4.1 \times 10^9$ 次浮点运算。
- 硬件算力：100 TFLOPS = $100 \times 10^{12}$ 次/秒。

理论推理时间：

$$
\text{时间} = \frac{\text{总 FLOPs}}{\text{硬件 FLOPS}}
$$

例如：

$$
4.1 \times 10^9 / 100 \times 10^{12} = 0.000041 \text{ 秒}
$$

但实际远不止，因为内存带宽、并行效率等问题。

---

# 9. 硬件算力 FLOPS 怎么来的？

以 GPU 为例：

$$
\text{峰值 FLOPS} = \text{核心数} \times \text{频率} \times \text{每周期操作数}
$$

比如一个 CUDA 核心每周期可以做 2 次 FP32 运算（一次 FMA），那么：

$$
\text{FP32 峰值} = \text{CUDA 核心数} \times \text{频率} \times 2
$$

Tensor Core 更高，比如 A100：

- FP16 Tensor Core 峰值约 312 TFLOPS。
- FP32 约 19.5 TFLOPS。

但这是**峰值**，实际利用率可能只有 30%~70%。

---

# 10. 为什么 FLOPs 低不一定快？

FLOPs 只是理论计算量。实际速度还受：

1. **内存带宽**：数据搬运可能比计算更慢。
2. **并行度**：GPU 需要足够多线程才能跑满。
3. **算子融合**：多个小算子合并成一个，减少内存访问。
4. **批大小**：batch 太小，GPU 利用率低。
5. **量化**：INT8 计算比 FP32 快，但 FLOPs 统计方式不同。
6. **稀疏化**：权重稀疏后，理论 FLOPs 降低，但硬件不一定支持。
7. **硬件架构**：有些硬件对卷积优化好，对注意力优化差。

这就是 **Roofline 模型** 说的：

- 计算强度 = FLOPs / 内存访问字节数。
- 计算强度低 → 内存瓶颈。
- 计算强度高 → 计算瓶颈。

---

# 11. 训练和推理的 FLOPs

## 11.1 推理

只做前向传播：

$$
\text{推理 FLOPs} = \text{前向 FLOPs}
$$

## 11.2 训练

训练包括：

- 前向传播：1 份
- 反向传播：约 2 份
- 参数更新：少量

所以训练总计算量约：

$$
\text{训练 FLOPs} \approx 3 \times \text{前向 FLOPs}
$$

如果用了激活重计算，前向会多算一次，总计算量可能增加到 4 倍左右。

---

# 12. 常用统计工具

PyTorch：

- `thop`
- `fvcore`
- `ptflops`
- `calflops`
- `torchstat`

TensorFlow：

- `tf.profiler`
- `keras-flops`

使用时注意：

- 是否包含 bias？
- 是否包含激活、BN？
- batch size 是多少？
- 输入分辨率是多少？
- 统计的是 MACs 还是 FLOPs？

不同工具结果可能差 2 倍，就是因为口径不同。

---

# 13. 常见模型 FLOPs 量级

| 模型 | 输入 | FLOPs |
|---|---|---|
| MobileNetV2 | 224×224 | ~0.3 GFLOPs |
| ResNet-50 | 224×224 | ~4.1 GFLOPs |
| VGG-16 | 224×224 | ~15.5 GFLOPs |
| BERT-base | 128 tokens | ~22 GFLOPs |
| GPT-3 175B | 每 token | ~350 GFLOPs |

这些数字只是量级，具体取决于实现和统计口径。

---

# 14. 怎么降低 FLOPs？

1. **轻量卷积**：深度可分离卷积、分组卷积。
2. **剪枝**：去掉不重要的权重。
3. **量化**：FP32 → FP16 → INT8，计算更快，但 FLOPs 定义会变。
4. **知识蒸馏**：小模型学大模型。
5. **低秩分解**：把大矩阵拆成小矩阵。
6. **注意力优化**：稀疏注意力、线性注意力。
7. **算子融合**：减少内存访问，不减少 FLOPs 但加速。

---

# 15. 总结：一张图记住

```
FLOPs（计算量） = 浮点运算次数
FLOPS（算力）   = 每秒浮点运算次数

基本规则：
1 MAC = 2 FLOPs
FLOPs = 2 × MACs

线性层：
FLOPs ≈ 2 × 权重参数量 × batch size
MACs  = 权重参数量 × batch size

卷积层：
FLOPs = 2 × B × H_out × W_out × C_out × C_in × K_h × K_w

训练：
总 FLOPs ≈ 3 × 前向 FLOPs

实际速度：
不是只看 FLOPs，还要看内存带宽、并行度、硬件支持。
```

一句话：

> **FLOPs 衡量模型要算多少次，FLOPS 衡量硬件每秒能算多少次。FLOPs 是理论计算量，不等于实际速度。**